# JEPA-Histo — Pipeline complet sur Google Colab

**Ce notebook couvre :**
1. Setup GPU + dépendances
2. Montage Google Drive (checkpoints persistants)
3. Clonage du repo
4. Téléchargement de PatchCamelyon
5. Préentraînement I-JEPA
6. Linear probe (100 % des labels)
7. Few-shot learning (1 %, 5 %, 10 %)
8. Visualisation t-SNE

> **Important** : aller dans `Environnement d'exécution → Modifier le type d'environnement d'exécution → GPU (T4 ou A100)`

## Cellule 1 — Vérification GPU

In [ ]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM totale     :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'Go')
else:
    print('⚠️  Aucun GPU détecté — aller dans Environnement d\'exécution → GPU')

## Cellule 2 — Montage Google Drive

Les checkpoints seront sauvegardés dans `Mon Drive/JEPA-Histo/outputs/`.
Si la session Colab se déconnecte, on pourra reprendre depuis le dernier checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/JEPA-Histo/outputs/jepa', exist_ok=True)
print('✅ Drive monté — checkpoints dans Mon Drive/JEPA-Histo/outputs/')

## Cellule 3 — Installation des dépendances

In [ ]:
%%capture
!pip install timm==0.9.16 h5py pyyaml scikit-learn matplotlib

## Cellule 4 — Clonage du repo

In [ ]:
import os

REPO_URL = 'https://github.com/tristanlecourtois/JEPA-Histo.git'  # ← mettre l'URL de votre repo
REPO_DIR = '/content/JEPA-Histo'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repo déjà présent — mise à jour effectuée.')

# Ajouter le repo au sys.path pour les imports
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print('Répertoire courant :', os.getcwd())

## Cellule 5 — Téléchargement PatchCamelyon (~7 Go)

Les fichiers sont téléchargés directement dans `/content/data/patchcamelyon/`.

> ⏱ ~5-10 min selon la connexion Colab.

In [ ]:
import os

DATA_DIR = '/content/data/patchcamelyon'
os.makedirs(DATA_DIR, exist_ok=True)

BASE = 'https://zenodo.org/record/2546921/files'

files = [
    'camelyonpatch_level_2_split_train_x.h5',
    'camelyonpatch_level_2_split_train_y.h5',
    'camelyonpatch_level_2_split_valid_x.h5',
    'camelyonpatch_level_2_split_valid_y.h5',
    'camelyonpatch_level_2_split_test_x.h5',
    'camelyonpatch_level_2_split_test_y.h5',
]

for fname in files:
    dest = f'{DATA_DIR}/{fname}'
    gz   = dest + '.gz'
    if os.path.exists(dest):
        print(f'✅ {fname} déjà présent')
        continue
    print(f'⬇️  Téléchargement {fname} ...')
    !wget -q --show-progress -O {gz} {BASE}/{fname}.gz
    !gunzip {gz}
    print(f'   → décompressé')

print('\n✅ PatchCamelyon prêt !')
!ls -lh {DATA_DIR}

## Cellule 6 — Vérification du dataset et du modèle

In [ ]:
import yaml
import torch
from datasets.patchcamelyon import PatchCamelyon
from models.ssl.jepa import IJEPA
from utils.seed import set_seed

set_seed(42)

# Charger la config Colab
with open('configs/jepa_colab.yaml') as f:
    cfg = yaml.safe_load(f)

# Vérifier le dataset
ds = PatchCamelyon('/content/data/patchcamelyon', split='train')
print(ds)
img, label = ds[0]
print(f'Image : {img.size}  |  Label : {label} ({["normal","tumour"][label]})')

# Vérifier le modèle — forward pass
device = torch.device('cuda')
model  = IJEPA(
    encoder_cfg   = cfg['model']['encoder'],
    predictor_cfg = cfg['model']['predictor'],
    masking_cfg   = cfg['masking'],
).to(device)

x   = torch.randn(2, 3, 96, 96, device=device)
out = model(x)
print(f'\n✅ Forward pass OK  |  loss = {out["loss"].item():.4f}')

total_params     = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Paramètres totaux    : {total_params:.1f} M')
print(f'Paramètres entraînables : {trainable_params:.1f} M')

## Cellule 7 — Préentraînement I-JEPA

**Durée estimée :**
- T4  (15 Go) + batch 128 + 100 époques ≈ 3-4 h
- A100 (40 Go) + batch 256 + 100 époques ≈ 1.5 h

Les checkpoints sont sauvegardés sur Drive toutes les 5 époques.  
Si la session se déconnecte, relancer cette cellule avec `--resume`.

In [ ]:
import os

RESUME = ''  # laisser vide pour un nouvel entraînement
# Pour reprendre : RESUME = '/content/drive/MyDrive/JEPA-Histo/outputs/jepa/checkpoint_latest.pth'

resume_flag = f'--resume {RESUME}' if RESUME and os.path.exists(RESUME) else ''

!python experiments/run_pretrain.py \
    --config configs/jepa_colab.yaml \
    {resume_flag}

## Cellule 8 — Vérifier les checkpoints sauvegardés

In [ ]:
!ls -lh /content/drive/MyDrive/JEPA-Histo/outputs/jepa/

## Cellule 9 — Linear Probe (100 % des labels)

Évalue les représentations apprises avec un classifieur linéaire entraîné sur toutes les données labelisées.

In [ ]:
CHECKPOINT = '/content/drive/MyDrive/JEPA-Histo/outputs/jepa/checkpoint_latest.pth'
# Remplacer par checkpoint_best.pth si disponible

!python experiments/run_linear_probe.py \
    --config  configs/jepa_colab.yaml \
    --checkpoint {CHECKPOINT}

In [ ]:
# Afficher les résultats
import json

results_path = '/content/drive/MyDrive/JEPA-Histo/outputs/jepa/linear_probe/results.json'
with open(results_path) as f:
    results = json.load(f)

print('=== Linear Probe Results ===')
print(f'  Accuracy : {results["acc"]*100:.2f} %')
print(f'  AUROC    : {results["auroc"]:.4f}')

## Cellule 10 — Few-Shot Learning

Entraîne un classifieur linéaire avec seulement 1 %, 5 %, 10 % des labels.  
C'est l'expérience principale du papier.

In [ ]:
CHECKPOINT = '/content/drive/MyDrive/JEPA-Histo/outputs/jepa/checkpoint_latest.pth'

!python experiments/run_fewshot.py \
    --config      configs/jepa_colab.yaml \
    --checkpoint  {CHECKPOINT} \
    --fractions   0.01 0.05 0.10 1.0 \
    --seeds       0 1 2

In [ ]:
# Afficher le tableau récapitulatif
import json

fewshot_path = '/content/drive/MyDrive/JEPA-Histo/outputs/jepa/fewshot/fewshot_results.json'
with open(fewshot_path) as f:
    results = json.load(f)

print('=== Few-Shot Results (mean ± std sur 3 seeds) ===')
print(f'{"Fraction":>10}  {"Accuracy":>20}  {"AUROC":>20}')
print('-' * 55)
for frac_key, v in results.items():
    acc_str   = f"{v['acc_mean']*100:.2f} ± {v['acc_std']*100:.2f} %"
    auroc_str = f"{v['auroc_mean']:.4f} ± {v['auroc_std']:.4f}"
    print(f'{frac_key:>10}  {acc_str:>20}  {auroc_str:>20}')

## Cellule 11 — Visualisation t-SNE des représentations

Permet d'inspecter qualitativement la structure de l'espace latent appris.

In [ ]:
import torch
import yaml
from torch.utils.data import DataLoader

from datasets.patchcamelyon import PatchCamelyon
from models.ssl.jepa import IJEPA
from evaluation.embeddings import embed_dataset, compute_tsne, plot_embeddings
from utils.transforms import build_eval_transform

with open('configs/jepa_colab.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda')

# Charger l'encoder
CHECKPOINT = '/content/drive/MyDrive/JEPA-Histo/outputs/jepa/checkpoint_latest.pth'

full_model = IJEPA(
    encoder_cfg   = cfg['model']['encoder'],
    predictor_cfg = cfg['model']['predictor'],
    masking_cfg   = cfg['masking'],
)
ckpt = torch.load(CHECKPOINT, map_location='cpu')
full_model.load_state_dict(ckpt['model'])
encoder = full_model.context_encoder.to(device)

# Dataset de test (2 000 patches pour la vitesse)
aug_cfg = cfg.get('augmentation', {})
mean    = aug_cfg.get('normalize', {}).get('mean', [0.7008, 0.5384, 0.6916])
std     = aug_cfg.get('normalize', {}).get('std',  [0.2350, 0.2774, 0.2128])
transform = build_eval_transform(96, mean, std)

test_ds  = PatchCamelyon('/content/data/patchcamelyon', split='test', transform=transform)

# Sous-échantillonner pour la vitesse
from torch.utils.data import Subset
import numpy as np
np.random.seed(42)
indices = np.random.choice(len(test_ds), size=2000, replace=False)
subset  = Subset(test_ds, indices)

loader  = DataLoader(subset, batch_size=256, shuffle=False, num_workers=2)

print('Extraction des embeddings ...')
embs, labels = embed_dataset(encoder, loader, device, normalise=True)
print(f'Embeddings : {embs.shape}')

print('Calcul t-SNE ...')
reduced = compute_tsne(embs, perplexity=30)

plot_embeddings(
    reduced, labels,
    class_names=['Normal', 'Tumour'],
    title='I-JEPA — PatchCamelyon test set (t-SNE)',
    save_path='/content/drive/MyDrive/JEPA-Histo/outputs/jepa/tsne.png',
)
print('✅ Figure sauvegardée sur Drive.')

## Cellule 12 — kNN probe (évaluation non-paramétrique rapide)

Alternative au linear probe : pas d'entraînement, juste un vote pondéré par similarité cosinus.  
Utile pour évaluer rapidement un checkpoint intermédiaire.

In [ ]:
from evaluation.embeddings import embed_dataset, knn_accuracy

# Extraire les features train et test
train_ds = PatchCamelyon('/content/data/patchcamelyon', split='train', transform=transform)

# Sous-échantillonner le train (50k patches suffisent pour kNN)
train_idx  = np.random.choice(len(train_ds), size=50000, replace=False)
train_sub  = Subset(train_ds, train_idx)
train_ldr  = DataLoader(train_sub, batch_size=256, shuffle=False, num_workers=2)
test_ldr2  = DataLoader(Subset(test_ds, np.arange(len(test_ds))), batch_size=256, num_workers=2)

print('Extraction features train ...')
train_embs, train_lbls = embed_dataset(encoder, train_ldr, device)

print('Extraction features test  ...')
test_embs,  test_lbls  = embed_dataset(encoder, test_ldr2, device)

for k in [10, 20, 50]:
    acc = knn_accuracy(train_embs, train_lbls, test_embs, test_lbls, k=k)
    print(f'  kNN (k={k:2d})  accuracy = {acc*100:.2f} %')

---

## Résumé des fichiers générés

Tous les résultats sont persistés sur Google Drive :

```
Mon Drive/JEPA-Histo/outputs/jepa/
├── checkpoint_ep0005.pth      ← checkpoints périodiques
├── checkpoint_latest.pth      ← toujours le plus récent
├── checkpoint_best.pth        ← meilleure perte
├── config.yaml                ← snapshot de la config
├── logs/
│   ├── experiment.log         ← log texte complet
│   └── tb/                    ← events TensorBoard
├── linear_probe/
│   └── results.json           ← acc + auroc full-labels
├── fewshot/
│   └── fewshot_results.json   ← tableau mean ± std
└── tsne.png                   ← visualisation embeddings
```